# Patent Emerging Radar — Kaggle Embedding Job

## Before you run
1. Click **Edit** (top right)
2. **Add your dataset:** right sidebar → **+ Add data** → **Your Datasets** → select the dataset containing the patent file
3. Go to **Settings → Accelerator → GPU T4 x1**
4. Run all cells — come back in ~2 hours

## What you get back
`embeddings.zip` in the Output panel. Download it and extract into:
```
patent-emerging-radar/data/processed/embeddings/
```
Then open the main notebook and continue from **Part 3**.

---
## Cell 1 — GPU check + install

In [ ]:
import subprocess, sys, os, time, pathlib

try:
    import torch
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'GPU: {name}  ({vram:.1f} GB VRAM)  ✓')
    else:
        print('WARNING: No GPU — go to Settings → Accelerator → GPU T4 x1')
except ImportError:
    print('PyTorch not available')

print('Installing sentence-transformers ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'],
               check=True)
print('Done. (Dependency conflict warnings above are harmless Kaggle noise — ignore them.)')

WORKDIR   = pathlib.Path('/kaggle/working')
EMBED_DIR = WORKDIR / 'embeddings'
EMBED_DIR.mkdir(exist_ok=True)
print(f'Embed dir: {EMBED_DIR}')

---
## Cell 2 — Find patent file + Configuration

Set `DATASET_DIR` to the folder Kaggle mounted your dataset at.
The script finds the `.tsv` or `.tsv.zip` inside it automatically.

In [ ]:
# ── ONLY LINE YOU NEED TO CHANGE ─────────────────────────────────────────────
DATASET_DIR = '/kaggle/input/datasets/peannut/patents1'
# ─────────────────────────────────────────────────────────────────────────────

YEAR_START   = 1996
YEAR_END     = 2025
BATCH_SIZE   = 512
MODEL_ID     = 'all-MiniLM-L6-v2'
WIPO_GRANTED = {'B2', 'B1'}

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# List what's actually in the folder
ds = pathlib.Path(DATASET_DIR)
print(f'Contents of {DATASET_DIR}:')
if ds.exists():
    for f in sorted(ds.rglob('*')):
        if f.is_file():
            print(f'  {f.name}  ({f.stat().st_size/1e6:.0f} MB)')
else:
    print('  NOT FOUND — check DATASET_DIR')

# Auto-find the patent file (largest .tsv or .tsv.zip in the folder)
def find_patent_file():
    if ds.exists():
        for ext in ['.tsv.zip', '.tsv']:
            hits = [f for f in ds.rglob('*') if f.name.endswith(ext) and f.is_file()]
            if hits:
                return max(hits, key=lambda f: f.stat().st_size)
    for name in ['g_patent.tsv.zip', 'g_patent.tsv']:
        hits = list(pathlib.Path('/kaggle/input').rglob(name))
        if hits: return hits[0]
    cached = WORKDIR / 'g_patent.tsv.zip'
    if cached.exists(): return cached
    return None

found = find_patent_file()
if found:
    PATENTS_FILE  = found
    IS_COMPRESSED = PATENTS_FILE.suffix == '.zip'
    print(f'\nUsing : {PATENTS_FILE}  ({PATENTS_FILE.stat().st_size/1e6:.0f} MB)')
    print(f'Type  : {"zip" if IS_COMPRESSED else "plain TSV"}  ✓')
else:
    print('\nFile not found in DATASET_DIR — downloading from PatentsView S3 ...')
    dest = WORKDIR / 'g_patent.tsv.zip'
    subprocess.run(['wget','-q','--show-progress',
        'https://s3.amazonaws.com/data.patentsview.org/download/g_patent.tsv.zip',
        '-O', str(dest)], check=True)
    PATENTS_FILE  = dest
    IS_COMPRESSED = True
    print(f'Downloaded {PATENTS_FILE.stat().st_size/1e6:.0f} MB  ✓')

print(f'Device: {device}  |  Years: {YEAR_START}–{YEAR_END}  |  Batch: {BATCH_SIZE}')

---
## Cell 3 — Embedding functions

In [ ]:
import csv, io, zipfile, numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

def stream_patents(file_path, year_start, year_end):
    """Handles both .tsv and .tsv.zip."""
    def rows(fh):
        return csv.DictReader(io.TextIOWrapper(fh, encoding='utf-8', errors='replace'),
                              delimiter='\t')
    def process(reader):
        for row in reader:
            if (row.get('wipo_kind') or '').strip() not in WIPO_GRANTED: continue
            date = (row.get('patent_date') or '').strip()
            if len(date) < 4: continue
            try: year = int(date[:4])
            except ValueError: continue
            if not (year_start <= year <= year_end): continue
            title = (row.get('patent_title') or '').strip()
            if title:
                yield row.get('patent_id',''), date, year, title
    p = pathlib.Path(file_path)
    if p.suffix == '.zip':
        with zipfile.ZipFile(p) as z:
            with z.open(z.namelist()[0]) as raw:
                yield from process(rows(raw))
    else:
        with open(p, 'rb') as raw:
            yield from process(rows(raw))

def fmt(s):
    if s < 60: return f'{s:.0f}s'
    if s < 3600: return f'{s/60:.1f}min'
    return f'{s/3600:.1f}h'

print('Functions ready.')

---
## Cell 4 — Load model

In [ ]:
print(f'Loading {MODEL_ID} ...')
t0 = time.time()
model = SentenceTransformer(MODEL_ID, device=device)
print(f'Model loaded in {fmt(time.time()-t0)}  |  running on: {device}')

---
## Cell 5 — Stream + bucket patents

Reads the full file once and sorts patents by year. ~5–10 min.

In [ ]:
done    = {int(f.stem) for f in EMBED_DIR.glob('*.npz') if f.stem.isdigit()}
buckets = defaultdict(list)
total   = 0
t0      = time.time()

if done: print(f'Already embedded (will skip): {sorted(done)}')
print(f'Streaming {PATENTS_FILE.name} ...')

with tqdm(desc='Reading', unit=' rows', unit_scale=True, mininterval=2) as bar:
    for pid, date, year, title in stream_patents(PATENTS_FILE, YEAR_START, YEAR_END):
        if year not in done:
            buckets[year].append((pid, date, year, title))
        total += 1
        bar.update(1)
        if total % 500_000 == 0:
            bar.set_postfix_str(f'kept={sum(len(v) for v in buckets.values()):,}')

kept = sum(len(v) for v in buckets.values())
print(f'\nRead {total:,} rows in {fmt(time.time()-t0)}')
print(f'Kept {kept:,} patents across {len(buckets)} years to embed')

---
## Cell 6 — Encode and save

Main GPU work. ~2 hours on T4. Each year saves to disk as it finishes — if the session dies, re-run from Cell 5 and already-done years are skipped.

In [ ]:
speed_log   = []
total_start = time.time()

for i, year in enumerate(sorted(buckets.keys())):
    out  = EMBED_DIR / f'{year}.npz'
    recs = buckets[year]
    n    = len(recs)
    if speed_log:
        avg  = sum(speed_log) / len(speed_log)
        rem  = sum(len(buckets[y]) for y in sorted(buckets)[i:])
        eta  = f'  (est. {fmt(rem/avg)} left)'
    else:
        eta = ''
    print(f'\n── {year}  {n:,} patents{eta}')
    t0  = time.time()
    emb = model.encode([r[3] for r in recs], batch_size=BATCH_SIZE,
                       show_progress_bar=True, normalize_embeddings=True,
                       convert_to_numpy=True, device=device).astype(np.float32)
    elapsed = time.time() - t0
    speed_log.append(n / elapsed)
    np.savez_compressed(out,
        ids=np.array([r[0] for r in recs]),
        dates=np.array([r[1] for r in recs]),
        titles=np.array([r[3] for r in recs]),
        embeddings=emb)
    mb = out.stat().st_size / 1e6
    print(f'   saved {mb:.0f} MB → {out.name}  [{fmt(elapsed)}, {n/elapsed:.0f}/sec]')

print(f'\n✓ All done in {fmt(time.time()-total_start)}')
files = sorted(EMBED_DIR.glob('*.npz'))
print(f'{len(files)} files  |  {sum(f.stat().st_size for f in files)/1e9:.2f} GB total')

---
## Cell 7 — Verify + zip for download

**Deletes each `.npz` immediately after adding to the zip** so you never hold both copies at once — keeps output well under the 19.5 GB limit.

In [ ]:
import zipfile as zf

npz_files = sorted(EMBED_DIR.glob('*.npz'))

# Spot check
if npz_files:
    d = np.load(npz_files[-1], allow_pickle=True)
    print(f'Spot check {npz_files[-1].stem}:')
    print(f'  shape  : {d["embeddings"].shape}')
    print(f'  norms  : {np.linalg.norm(d["embeddings"][:5], axis=1).round(4)}  (should be ~1.0)')
    print(f'  sample : {d["titles"][0]}')
    del d

# Zip and delete originals one at a time to stay under 19.5 GB
zip_path = WORKDIR / 'embeddings.zip'
print(f'\nZipping {len(npz_files)} files (deleting originals as we go to save space) ...')

with zf.ZipFile(zip_path, 'w', compression=zf.ZIP_STORED) as zout:
    for f in npz_files:
        mb = f.stat().st_size / 1e6
        zout.write(f, arcname=f.name)
        f.unlink()   # free space immediately
        used = zip_path.stat().st_size / 1e9
        print(f'  + {f.name}  ({mb:.0f} MB)  →  zip: {used:.2f} GB')

zip_mb = zip_path.stat().st_size / 1e6
print(f'\nembeddings.zip  {zip_mb:.0f} MB  ✓')
print()
print('── Download ───────────────────────────────────────────────────────────')
print('Right sidebar → Output → embeddings.zip → click the download arrow')
print('Extract into:  patent-emerging-radar/data/processed/embeddings/')
print('Then open the main notebook and continue from Part 3.')

---
## Done!

Verify locally after extracting:
```python
import numpy as np, pathlib
EMBED_DIR = pathlib.Path('../data/processed/embeddings')
files = sorted(EMBED_DIR.glob('*.npz'))
print(f'{len(files)} year files: {[f.stem for f in files]}')
d = np.load(files[-1], allow_pickle=True)
print(d['embeddings'].shape)   # should be (N, 384)
```
Then run the main notebook from **Part 3** — skip Part 2 entirely.